## Analysis of experimental data (October 2023)

Parsing of the logfiles generated during the October 2023 experiments.


### Experiment Classes

There are two classes of experiment:

- RPE
- Cognitive Bias (CB)

The RF have the following subtypes:

- RPE-A= acquisition of response
- RPE-H= habit formation
- RPE-E= extinction of response
- RPE-R= reinstatement of response

### RPE-type experiments

A new experiment type was created during the experiments which was not in the original specification.
Can extract experiment details from Comment field. TODO: CH to confirm details.

#### Logfile Exclusions

Logfiles that are from test runs and also bad data need to be excluded from the analysis.

Rules are **case-insensitive**. Files which satisfy the following conditions are excluded:

- All files with `_TEST_` as the subject name
- All files with `_FRECKLE_` as the subject name
- Some files with `_BONNIE_` (14 legitimate files - TODO: CH to confirm details)
- All files with `test` in the `Comment` field
- Possibly some logs with very short run-times - TODO: CH to confirm the files that have been identified
- All files with `_OLIVE_` as the subject name - probably discard, treat as optional for now (TODO: CH to confirm treatment)

#### Problems with log file names during experiments

- Appears that for greater than 20 trials (subjects?) the value in the filename was reported as `NaN`. e.g. see `pumba` experiments.
- Other `NaN`s? There are 31 files with `NaN` (some are restarts) - TODO: Confirm with CH why were there restarts? Why not "new" experiment?

#### Time differences

For each trial we calculate the following time differences: 

1. Time delta: (`touch` datetime - `start tone` datetime) 
2. Time delta: (`Next start` datetime - `dispense of pellets` datetime)
3. Time delta: (`Dispense final pellets` datetime - `start tone` datetime)

We use item 3 as cross check on the consistency of previous time deltas.

These calculated quantities are the same for all RFE-type experiments.

### Cognitive Bias Experiments

- Extract "RIGHT" or "LEFT" from the Comment field.
- Also extract details from log file name? TODO: Confirm CH
- 1. Training experiments: 
  - Type 1
  - Type 2
-  2. Testing experiments: 
   -  Type 1
   -  Type 2
   -  Type 3
   -  Type 4 (re-uses Type 1 with indicator to distinguish in Comment field)

#### Time differences

For each trial: 

Training Type 1 (randomised versus fixed):
- `Start` datetime = Green button pressed and horse is released
- Capture positive (`GO`) / negative (`NOGO`) response time subject to maximum cutoff time (e.g. 30 seconds)
- In addition to left/right positioning of feed, there are also median, near positive and near negative positions.

TODO check with CH: Test only - be in all?

TODOs:
- create lists of various filenames (include, exclude, etc) for CH to review.

In [ ]:
# | echo: false

from pathlib import Path

import duckdb
import pandas as pd
from horse_logic.logfiles import Logs, Logfile
from logparser.Drain import LogParser
from tqdm.notebook import tqdm

In [ ]:
# | echo: false

from loguru import logger
from itables import init_notebook_mode

init_notebook_mode(all_interactive=True)  # for displaying dataframes in a more friendly manner

In [ ]:
# | echo: false

path_logfiles = "../data/results/data_17Jan2020_email_hillydale_equine"
assert Path(path_logfiles).exists()

In [ ]:
# | echo: false

logs = Logs(path=path_logfiles, patterns=["*_test*.log", "*freckle*.log"], include=False)

In [ ]:
# | echo: false

print(logs)

print(f"Included logfiles: {len(logs.included_files)}")
print(f"Excluded logfiles: {len(logs.excluded_files)}")

In [ ]:
# | echo: false

exclude_file_list = ["file1_exclude.log", "file2_exclude.log"]
logs.handle_specific_files(exclude_file_list, include=False)

In [ ]:
# | echo: false

df_excluded = pd.DataFrame(logs.excluded_files, columns=["Excluded"])

In [ ]:
df_excluded

In [ ]:
# | echo: false

df_included = pd.DataFrame(logs.included_files, columns=["Included"])

In [ ]:
df_included

In [ ]:
# | echo: false

df_file_components = logs.create_filename_components_dataframe(logs.included_files)

In [ ]:
# | echo: false

df_out = df_file_components.sort_values(by=["subject_name", "session_number"]).reset_index(drop=True)

In [ ]:
df_out

In [ ]:
df_out.drop(labels=["suffix"], axis=1, inplace=True)

In [ ]:
df_out = df_out[["original_filename", "subject_name", "experiment_type", "session_number", "datetime"]]

In [ ]:
df_out

In [ ]:
df_out["time_diff"] = df_out["datetime"].diff()

In [ ]:
df_out

In [ ]:
df_out["session_number"] = df_out["session_number"].astype("Int64")

In [ ]:
df_bonnie = df_out[df_out["subject_name"] == "bonnie"]

In [ ]:
df_bonnie[df_bonnie["time_diff"] < pd.Timedelta("5m")]

In [ ]:
bonnie_exclude_list = df_bonnie[df_bonnie["time_diff"] < pd.Timedelta("5m")]["original_filename"].tolist()

In [ ]:
len(bonnie_exclude_list)

In [ ]:
logs.handle_specific_files(bonnie_exclude_list, include=False)

In [ ]:
len(logs.excluded_files)

In [ ]:
# !open $path_logfiles/Experiment_2023-10-01T21:30:25.370452_bonnie_1_RPE-ER.log

In [ ]:
# !open $path_logfiles/Experiment_2023-10-02T21:29:38.930298_bonnie_3_RPE-E.log

In [ ]:
# !open $path_logfiles/Experiment_2023-10-03T14:46:54.142393_bonnie_13_RPE-A.log

In [ ]:
# !open $path_logfiles/Experiment_2023-10-19T14:37:56.973616_bonnie_72_RPE-H.log

In [ ]:
# | echo: false

df_out.to_csv("/Users/mjboothaus/code/github/databooth/horse-logic/data/results/experiment_summary_Oct2023_included.csv")

#### Try regex parsing of files

In [ ]:
logfile_eg = f"{logs.path}/Experiment_2023-11-17T05:11:53.655051_clover_nan_RPE-R.log"

In [ ]:
logfile_eg

In [ ]:
log_eg = Logfile(logfile_eg)

In [ ]:
log_eg.parse_filename_components()

```
class Experiment:
    ExperimentID: int
    SubjectName: str
    SessionNumber: int
    ExperimentType: str
    Comment: Optional[str]
    DateTime: datetime
    LogFileName: str
```

In [ ]:
log_eg.parse_comments()

In [ ]:
log_eg.parse_parameters()

For each parameter_name, parameter_value in parameters

```
class Parameter:
    ParameterID: int
    ExperimentID: int
    ParameterName: str
    ParameterValue: str
```

In [ ]:
from pprint import pprint

In [ ]:
pprint(log_eg.parse_trials_and_events())

In [ ]:
n_trial = len(log_eg.parse_trials_and_events()[0])

In [ ]:
n_trial

In [ ]:
# To get better parsing results, you are suggested to tune the hyper-parameters `st` and `depth`.

# input_dir = '../data/test_log/' # The input directory of log file
# output_dir = 'result/'  # The output directory of parsing results
# log_file = 'unknown.log'  # The input log file name

log_format = '<Date> <Time> <Level>:<Content>' # Define log format to split message fields
# Regular expression list for optional preprocessing (default: [])

#regex = [
#    r'(/|)([0-9]+\.){3}[0-9]+(:[0-9]+|)(:|)' # IP
#]

st = 0.5  # Similarity threshold
depth = 4  # Depth of all leaf nodes

parser = LogParser(log_format, logs.path, outdir="./parse",  depth=depth, st=st, rex=[])

In [ ]:
!cat $logfile_eg

In [ ]:
parse_result = {}
for logfile in tqdm(logs.included_files, total=len(logs.included_files)):
    try:
        parse_result[Path(logfile).name] = parser.parse(Path(logfile).name)
    except Exception as e:
        print(f"ERROR: Unable to process {Path(logfile).name}")
        print(e)

In [ ]:
parse_result

In [ ]:
# con = duckdb.connect(database="../experiments_Oct2023.ddb")

In [ ]:
# con.query("SELECT * FROM Experiments").fetchdf()